# Tier 3 (v2) — Transformers em N = 200 mil (2 T4)

Protocolo de 2026-10-16 (`docs/PLANO_FECHAMENTO.md`, §"VERSÃO 2 — Tier 3"; runner `scripts/run_tier3.py`):

* Bases **HIGGS500K, COVER, KDD99**, subamostradas a 200 mil (160 mil de treino, 40 mil de teste). Os três parquets estão no repositório — nada é baixado.
* **Seleção única por base**: Optuna TPE (semente 0), **20 trials** por (modelo, base), hold-out 80/20 do treino da semente 0. `lr` entra na busca (contínua, 1e-4 a 3e-3).
* Orçamento **200 épocas / paciência 16**, também no ajuste final, restaurando o melhor checkpoint. Sem pruner: a paciência já interrompe cada trial.
* FT-CUR com **m = 2000** fixo (≈1% de N), igual aos Nyström e ao PCP.
* **Ajuste final**: 10 sementes, hiperparâmetros congelados.

**Não cabe em uma sessão de 12 h** (estimativa ≈7 h de seleção + ≈9–10 h de ajuste final). Por isso:

| sessão | células | o que faz |
|---|---|---|
| 0 | 1–5 | **piloto**: 1 trial por modelo no HIGGS500K → projeção de tempo (≈30–60 min) |
| 1 | 1–4, 6 | seleção (retomável) |
| 2 | 1–4, 7 | ajuste final (retomável) |

Tudo é retomável: ao fim (ou se a sessão cair), baixe `/kaggle/working/tier3_out/`, suba como *Dataset* e aponte `RESUME_DIR` na célula 4.

**Antes de rodar:** Settings → Accelerator → **GPU T4 x2**; Internet **ligada** (para o `git clone`).

In [ ]:
# ── 1. GPUs ──
import torch
print('CUDA:', torch.cuda.is_available(), '| placas:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()): print(' ', i, torch.cuda.get_device_name(i))
assert torch.cuda.device_count() >= 2, 'Settings → Accelerator → GPU T4 x2'

In [ ]:
# ── 2. Repositório ──
import os, subprocess, sys
GIT_URL     = 'https://github.com/PauloBernardo/dissertacao-estudo-comparativo.git'
PROJECT_DIR = '/kaggle/working/sparse-lssvm-transformers-study'
BRANCH      = 'revisao/estatistica-e-proveniencia'
if os.path.exists(PROJECT_DIR):
    subprocess.run(['git', '-C', PROJECT_DIR, 'fetch', 'origin'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, GIT_URL, PROJECT_DIR], check=True)
subprocess.run(['git', '-C', PROJECT_DIR, 'checkout', BRANCH], check=True)
subprocess.run(['git', '-C', PROJECT_DIR, 'pull', '--rebase', 'origin', BRANCH], check=True)
os.chdir(PROJECT_DIR)
!git log --oneline -2
sys.path.insert(0, '.')
assert os.path.exists('scripts/run_tier3.py'), 'clone sem run_tier3.py — refaça o pull'

In [ ]:
# ── 3. Dependências e dados (em cache no clone) ──
!pip install -q einops optuna 2>&1 | tail -n 1
from pathlib import Path
from src.data.loaders import DatasetLoader
TIER3 = ['HIGGS500K', 'COVER', 'KDD99']
falt = [d for d in TIER3 if not Path(f'data/raw/{d}.parquet').exists()]
assert not falt, f'sem parquet no clone: {falt} — refaça o pull'
for d in TIER3:
    X, y, _ = DatasetLoader.load(d)
    print(f'{d:10s} {X.shape}  positivos {y.mean():.3f}')

In [ ]:
# ── 4. Configuração, retomada e execução nas duas placas ──
import json, shutil, time, subprocess, glob
from pathlib import Path

OUTDIR = Path('/kaggle/working/tier3_out'); OUTDIR.mkdir(exist_ok=True)
RESUME_DIR = None    # ex.: Path('/kaggle/input/tier3-sessao1') para continuar de outra sessão
if RESUME_DIR:
    for f in Path(RESUME_DIR).rglob('tier3_*'):
        if not (OUTDIR / f.name).exists():
            shutil.copy(f, OUTDIR / f.name); print('retomado:', f.name)

DATASETS = 'HIGGS500K COVER KDD99'
SEEDS    = list(range(10))
# Divisão por placa: o SAINT (atenção B×B) é o mais caro, então fica com o FT-CUR,
# e os quatro FT dividem a outra. Ajuste com o resultado do piloto, se preciso.
GPU_MODELS = {0: ['FTTransformer_softmax', 'FTTransformer_topk', 'FTTransformer_entmax', 'FTTransformer_sparsemax'],
              1: ['SAINTColnorm', 'FTTransformerCURRandom']}

def files(gpu, tag=''):
    return (f'--selection-file {OUTDIR}/tier3_selection{tag}_gpu{gpu}.json '
            f'--final-file {OUTDIR}/tier3_final{tag}_gpu{gpu}.json '
            f'--storage sqlite:///{OUTDIR}/tier3_optuna{tag}_gpu{gpu}.db')

def run_2gpu(make_cmd, name):
    """Um processo por placa, com log próprio; espera os dois e mostra o fim dos logs."""
    procs = []
    for gpu in (0, 1):
        log = open(OUTDIR / f'tier3_{name}_gpu{gpu}.log', 'a')
        cmd = f'CUDA_VISIBLE_DEVICES={gpu} python -u ' + make_cmd(gpu)
        print(f'[gpu{gpu}]', cmd)
        procs.append((gpu, subprocess.Popen(cmd, shell=True, stdout=log, stderr=subprocess.STDOUT), log))
    t0 = time.time()
    while any(p.poll() is None for _, p, _ in procs):
        time.sleep(300)
        print(f'{(time.time()-t0)/3600:.2f} h —',
              ' | '.join(f'gpu{g}: ' + (open(l.name).read().strip().splitlines() or [''])[-1][-110:]
                         for g, _, l in procs))
    for g, p, l in procs:
        l.close(); print(f'gpu{g} terminou com código {p.returncode}')

In [ ]:
# ── 5. SESSÃO 0 — PILOTO: 1 trial por modelo no HIGGS500K ──
# Arquivos separados (tag _pilot): não contaminam a seleção de verdade.
run_2gpu(lambda g: 'scripts/run_tier3.py select --variants ' + ' '.join(GPU_MODELS[g]) +
         ' --datasets HIGGS500K --n-trials 1 ' + files(g, '_pilot'), 'pilot')

import optuna
proj = {}
for g in (0, 1):
    for v in GPU_MODELS[g]:
        st = optuna.load_study(study_name=f'{v}__HIGGS500K', storage=f'sqlite:///{OUTDIR}/tier3_optuna_pilot_gpu{g}.db')
        t = st.trials[0]
        proj.setdefault(g, 0.0)
        # seleção: 20 trials × 3 bases (128 mil); final: 10 sementes × 3 bases (160 mil ≈ 1,25×)
        sel_h, fin_h = 20*3*t.user_attrs['fit_s']/3600, 10*3*1.25*t.user_attrs['fit_s']/3600
        proj[g] += sel_h + fin_h
        print(f'gpu{g} {v:26s} trial {t.user_attrs["fit_s"]/60:6.1f} min  F1val {t.value:.4f}'
              f'  → seleção {sel_h:5.1f} h, final {fin_h:5.1f} h')
for g, h in proj.items(): print(f'gpu{g}: total projetado {h:.1f} h')
print('Se alguma placa passar de ~11 h em uma das fases, rebalanceie GPU_MODELS ou divida a fase em mais sessões.')

In [ ]:
# ── 6. SESSÃO 1 — seleção (20 trials por modelo × base; retomável) ──
run_2gpu(lambda g: 'scripts/run_tier3.py select --variants ' + ' '.join(GPU_MODELS[g]) +
         ' --datasets ' + DATASETS + ' ' + files(g), 'select')
for g in (0, 1):
    f = OUTDIR / f'tier3_selection_gpu{g}.json'
    if f.exists():
        for v, by in json.load(open(f)).items():
            for d, r in by.items(): print(f'{v:26s} {d:10s} F1val {r["best_val_f1"]:.4f}  {r["params"]}')
print('\nFim da sessão 1: baixe /kaggle/working/tier3_out/.')

In [ ]:
# ── 7. SESSÃO 2 — ajuste final (10 sementes, paciência 16; retomável) ──
run_2gpu(lambda g: 'scripts/run_tier3.py final --variants ' + ' '.join(GPU_MODELS[g]) +
         ' --datasets ' + DATASETS + ' --seeds ' + ' '.join(map(str, SEEDS)) + ' ' + files(g), 'final')
import collections, statistics as stt
recs = []
for g in (0, 1):
    f = OUTDIR / f'tier3_final_gpu{g}.json'
    if f.exists(): recs += json.load(open(f))
by = collections.defaultdict(list)
for r in recs:
    if r.get('status') == 'ok': by[(r['variant'], r['dataset'])].append(r['test_f1_macro'])
for (v, d), xs in sorted(by.items()):
    print(f'{v:26s} {d:10s} n={len(xs):2d}  F1 {stt.mean(xs):.4f} ± {stt.pstdev(xs):.4f}')
erros = [r for r in recs if r.get('status') != 'ok']
print(f'{len(erros)} erros', [ (r['variant'], r['dataset'], r['seed'], r['error'][:80]) for r in erros][:5])
print('\nFim: baixe /kaggle/working/tier3_out/.')